# 03 Substitution and equity

We aggregate demand point results from `02_network_accessibility` to block groups and examine who the Open Streets offset reached by borough, income, race, age, and tenure.

## Measures

We report three measures, since they order block groups differently. Offset rate is the share of loss recovered by 23 May, the last wave inside the closure window, and we calculate it as a ratio of population-weighted means within each group rather than a mean of block group ratios, which explode where loss is near zero. Zero offset marks block groups that recovered nothing, which hold 25.1% of the analyzed population. Better off marks block groups whose access ended above baseline, where a corridor arrived and no nearby playground closed, which hold 4.8% of the analyzed population.

## Exclusions

We exclude 307,959 people (3.5%) with no park or playground within 800 m at baseline, who could neither lose nor recover access, and a further 48,877 people (0.6%) in 39 block groups with park access but no playground within reach, whom the closure did not affect. We analyze the remaining 8,442,101 people in 6,380 block groups.

## Unit

We analyze block groups to match ACS publication, and the aggregation check in `02` finds citywide and borough offsets unchanged from demand point to tract.


In [2]:
import json
import os
import sys
from urllib.parse import urlencode, quote

import numpy as np
import pandas as pd
import requests
import statsmodels.formula.api as smf

sys.path.append("../src")
from paths import RAW, PROCESSED

pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

CENSUS_API_KEY = os.environ.get("CENSUS_API_KEY")
ACS_YEAR = 2020
ACS_CSV = RAW / f"acs{ACS_YEAR}_bg_nyc.csv"

STATE_FIPS = "36"
NYC_COUNTIES = {"005": "Bronx", "047": "Brooklyn", "061": "Manhattan",
                "081": "Queens", "085": "Staten Island"}
PLUTO_BORO = {"BK": "Brooklyn", "MN": "Manhattan", "QN": "Queens",
              "SI": "Staten Island", "BX": "Bronx"}

meta = json.load(open(PROCESSED / "2sfca_meta.json"))
WKEY = [d[5:].replace("-", "") for d in meta["waves"]]
LVL = ["baseline", "closure"] + [f"os_{d}" for d in WKEY]

In [3]:
def wmean(v, w):
    v, w = np.asarray(v, float), np.asarray(w, float)
    m = np.isfinite(v) & np.isfinite(w)
    return np.average(v[m], weights=w[m]) if m.any() and w[m].sum() > 0 else np.nan

def quintile_rates(df, var, q=5, n_boot=2000, seed=0):
    d = df[df[var].notna() & (df["pop"] > 0)].copy()
    d["q"] = pd.qcut(d[var], q, labels=range(1, q + 1))
    rng = np.random.default_rng(seed)

    out = []
    for qi, g in d.groupby("q", observed=True):
        w = g["pop"].values
        b, c = g["baseline"].values, g["closure"].values
        o = g[f"os_{WKEY[-1]}"].values

        wb, wc, wo = ((x * w).sum() / w.sum() for x in (b, c, o))
        r = {"q": int(qi), "n_bg": len(g), "pop": w.sum(), var: (g[var] * w).sum() / w.sum(),
             "pct_loss": 100 * (wb - wc) / wb,
             "offset_pct": 100 * (wo - wc) / (wb - wc),
             "zero_offset_pct": 100 * (g["zero_offset"] * w).sum() / w.sum(),
             "better_off_pct": 100 * (g["better_off"] * w).sum() / w.sum()}

        idx = rng.integers(0, len(g), (n_boot, len(g)))
        ws = w[idx]
        bb, cc, oo = ((x[idx] * ws).sum(1) / ws.sum(1) for x in (b, c, o))
        r["offset_lo"], r["offset_hi"] = np.percentile(100 * (oo - cc) / (bb - cc), [2.5, 97.5])
        out.append(r)

    return pd.DataFrame(out).set_index("q")

---
## 1. Accessibility results

We remove demand points with no baseline access and aggregate the competition-weighted results to block groups by population-weighted mean. We then remove block groups with no playground within reach of any resident.

In [5]:
tab = pd.read_parquet(PROCESSED / "2sfca_waves_competition.parquet")
CITY_POP = tab["pop_ceds"].sum()

keep = tab["baseline"] > 0
print(f"{len(tab):,} demand points  |  {CITY_POP:,.0f} people")
print(f"excluded, no baseline access: {(~keep).sum():,} demand points "
      f"({tab.loc[~keep, 'pop_ceds'].sum():,.0f} people, "
      f"{100 * tab.loc[~keep, 'pop_ceds'].sum() / CITY_POP:.1f}%)")

tab = tab[keep]

769,601 demand points  |  8,798,936 people
excluded, no baseline access: 44,755 demand points (307,959 people, 3.5%)


In [6]:
g = tab.groupby("bg_geoid")
bg = g.apply(lambda x: pd.Series({c: wmean(x[c], x["pop_ceds"]) for c in LVL}),
             include_groups=False).reset_index()
bg["pop"] = g["pop_ceds"].sum().values
bg["Borough"] = g["Borough"].first().map(PLUTO_BORO).values

bg["loss"] = bg["baseline"] - bg["closure"]
for d in WKEY:
    bg[f"offset_{d}"] = bg[f"os_{d}"] - bg["closure"]

print(f"{len(bg):,} block groups, {bg['pop'].sum():,.0f} people")

6,419 block groups, 8,490,978 people


In [7]:
bg["loss"] = bg["loss"].round(10)                    # remove floating-point noise
nl = bg["loss"] <= 0

NO_PLAYGROUND_BG = int(nl.sum())
NO_PLAYGROUND_POP = float(bg.loc[nl, "pop"].sum())

print(f"park access but no playground in reach: {NO_PLAYGROUND_BG} block groups, "
      f"{NO_PLAYGROUND_POP:,.0f} people ({100 * NO_PLAYGROUND_POP / CITY_POP:.1f}% of the city)")

bg = bg[~nl].reset_index(drop=True)
print(f"{len(bg):,} block groups remain, {bg['pop'].sum():,.0f} people")

park access but no playground in reach: 39 block groups, 48,877 people (0.6% of the city)
6,380 block groups remain, 8,442,101 people


---
## 2. Population characteristics

We join ACS 2016 to 2020 five-year estimates at the block group. Median household income at this level carries large margins of error, so we report its coefficient of variation, since a noisy covariate attenuates the gradient toward zero. We include under-18 share, since the closure targeted playgrounds.

In [9]:
# B09001 is not published at block group; build under-18 from the sex-by-age table
U18 = [f"B01001_{n:03d}E" for n in (3, 4, 5, 6, 27, 28, 29, 30)]

ACS_VARS = {
    "B03002_001E": "pop_total", "B03002_003E": "nh_white",
    "B19013_001E": "med_income", "B19013_001M": "med_income_moe",
    "B25003_001E": "tenure_total", "B25003_003E": "renter",
    **{v: v for v in U18},
}
OUT_COLS = ["pop_total", "nh_white", "med_income", "med_income_moe",
            "tenure_total", "renter", "under18"]

if ACS_CSV.exists():
    acs = pd.read_csv(ACS_CSV, dtype={"bg_geoid": str})
else:
    frames = []
    for fips, name in NYC_COUNTIES.items():
        params = {"get": ",".join(ACS_VARS), "for": "block group:*",
                  "in": f"state:{STATE_FIPS} county:{fips} tract:*"}
        if CENSUS_API_KEY:
            params["key"] = CENSUS_API_KEY
        url = f"https://api.census.gov/data/{ACS_YEAR}/acs/acs5?{urlencode(params, quote_via=quote, safe=':*,')}"

        r = requests.get(url, timeout=180)
        r.raise_for_status()
        rows = r.json()
        frames.append(pd.DataFrame(rows[1:], columns=rows[0]))
        print(f"{name:14} {len(rows)-1:>6,} block groups")

    acs = pd.concat(frames, ignore_index=True).rename(columns=ACS_VARS)
    acs["bg_geoid"] = acs["state"] + acs["county"] + acs["tract"] + acs["block group"]
    for c in ACS_VARS.values():
        acs[c] = pd.to_numeric(acs[c], errors="coerce")

    acs["under18"] = acs[U18].sum(axis=1)
    acs.loc[acs["med_income"] < 0, ["med_income", "med_income_moe"]] = np.nan   # negative values mark suppressed estimates
    acs = acs[["bg_geoid"] + OUT_COLS]
    acs.to_csv(ACS_CSV, index=False)

acs["pct_nonwhite"] = 100 * (1 - acs["nh_white"] / acs["pop_total"].replace(0, np.nan))
acs["pct_under18"] = 100 * acs["under18"] / acs["pop_total"].replace(0, np.nan)
acs["pct_renter"] = 100 * acs["renter"] / acs["tenure_total"].replace(0, np.nan)
acs["income_cv"] = (acs["med_income_moe"] / 1.645) / acs["med_income"]

print(f"{len(acs):,} block groups")
print(f"income missing: {acs['med_income'].isna().sum():,}")
print(f"income CV: median {acs['income_cv'].median():.2f}, p90 {acs['income_cv'].quantile(.9):.2f}")
print(f"under 18: median {acs['pct_under18'].median():.1f}%")

6,807 block groups
income missing: 1,102
income CV: median 0.24, p90 0.47
under 18: median 19.6%


In [10]:
ACS_COLS = ["med_income", "med_income_moe", "income_cv",
            "pct_nonwhite", "pct_under18", "pct_renter"]

bg = bg.drop(columns=ACS_COLS, errors="ignore")
bg = bg.merge(acs[["bg_geoid"] + ACS_COLS], on="bg_geoid", how="left")

miss = bg["med_income"].isna()
print(f"missing income: {miss.sum()} block groups, {bg.loc[miss, 'pop'].sum():,.0f} people "
      f"({100 * bg.loc[miss, 'pop'].sum() / bg['pop'].sum():.1f}%)")
print(f"missing race: {bg['pct_nonwhite'].isna().sum()}  |  child share: {bg['pct_under18'].isna().sum()}  |  "
      f"tenure: {bg['pct_renter'].isna().sum()}")

missing income: 748 block groups, 747,913 people (8.9%)
missing race: 84  |  child share: 84  |  tenure: 105


---
## 3. Measures

We calculate loss and offset as levels, with zero offset and better off as indicators. Citywide loss is 26.8% in the block group sample against 26.6% at the demand point in `02`, since the sample excludes block groups with no playground to lose.

In [12]:
bg["pct_loss"] = 100 * bg["loss"] / bg["baseline"]
bg["zero_offset"] = bg[f"offset_{WKEY[-1]}"] <= 0
bg["better_off"] = bg[f"os_{WKEY[-1]}"] > bg["baseline"]

w = bg["pop"]
wb, wc, wo = (np.average(bg[c], weights=w) for c in ["baseline", "closure", f"os_{WKEY[-1]}"])
print(f"citywide loss: {100*(wb-wc)/wb:.2f}%  |  offset: {100*(wo-wc)/(wb-wc):.2f}% of loss")
print(f"zero offset: {100*np.average(bg['zero_offset'], weights=w):.1f}% of population")
print(f"better off:  {100*np.average(bg['better_off'], weights=w):.1f}% of population")

citywide loss: 26.84%  |  offset: 10.19% of loss
zero offset: 25.1% of population
better off:  4.8% of population


---
## 4. Gradients

We calculate population-weighted rates by quintile from aggregated levels, with 95% bootstrap intervals on offset. Each gradient uses the block groups with a published value for its variable, and income excludes the 748 block groups without a published median.

In [14]:
print(quintile_rates(bg, "pct_nonwhite").round(2).to_string())

   n_bg           pop  pct_nonwhite  pct_loss  offset_pct  zero_offset_pct  better_off_pct  offset_lo  offset_hi
q                                                                                                               
1  1260 1,556,557.900        18.810    23.770      11.510           26.830           3.830     10.090     13.010
2  1259 1,650,513.100        46.600    23.140      13.790           26.770           5.410     11.940     16.130
3  1259 1,717,326.600        74.680    26.180      10.490           26.730           5.270      9.540     11.500
4  1259 1,832,450.030        93.170    28.920       7.670           25.630           4.780      6.900      8.450
5  1259 1,682,317.020        99.350    33.430       8.320           19.840           4.440      7.440      9.320


In [15]:
print(quintile_rates(bg, "med_income").round(2).to_string())

   n_bg           pop  med_income  pct_loss  offset_pct  zero_offset_pct  better_off_pct  offset_lo  offset_hi
q                                                                                                             
1  1127 1,698,225.610  30,742.150    34.400       7.390           16.470           2.540      6.700      8.070
2  1126 1,601,388.610  54,165.310    26.450       9.430           21.290           4.370      8.400     10.540
3  1126 1,522,684.010  71,923.240    23.390       9.530           34.090           5.270      8.330     10.800
4  1126 1,439,523.940  93,206.840    21.500       9.520           36.800           5.200      8.230     10.920
5  1127 1,432,365.960 153,111.290    24.640      16.770           22.340           7.420     14.590     19.350


In [16]:
print(quintile_rates(bg, "pct_renter").round(2).to_string())

   n_bg           pop  pct_renter  pct_loss  offset_pct  zero_offset_pct  better_off_pct  offset_lo  offset_hi
q                                                                                                             
1  1255 1,439,341.040      20.790    13.950       5.740           58.890           5.310      4.470      7.320
2  1255 1,613,561.680      48.630    24.120       9.900           34.040           5.250      8.600     11.380
3  1255 1,726,183.830      69.190    29.600      12.610           20.220           5.910     10.730     15.090
4  1255 1,832,924.490      86.350    33.960      12.230           10.680           5.530     11.210     13.360
5  1255 1,816,909.590      98.660    36.130       9.510            9.800           2.020      8.860     10.210


In [17]:
print(quintile_rates(bg, "pct_under18").round(2).to_string())

   n_bg           pop  pct_under18  pct_loss  offset_pct  zero_offset_pct  better_off_pct  offset_lo  offset_hi
q                                                                                                              
1  1260 1,582,150.280        6.890    32.440      13.700           14.100           6.770     12.410     15.050
2  1259 1,668,352.720       14.540    26.560      12.450           25.180           5.020     10.710     14.720
3  1259 1,706,348.540       19.530    25.200       9.370           29.080           4.970      8.300     10.500
4  1259 1,748,236.180       24.550    23.190       8.570           30.840           4.590      7.550      9.700
5  1259 1,734,076.930       34.060    27.420       7.220           25.570           2.640      6.430      8.070


In [18]:
for v in ["pct_under18", "med_income"]:
    d = bg[bg[v].notna()].copy()
    d["q"] = pd.qcut(d[v], 5, labels=range(1, 6))
    print(f"\n{v}, population share by borough within quintile:")
    print(pd.crosstab(d["q"], d["Borough"], values=d["pop"], aggfunc="sum",
                      normalize="index").round(3).to_string())


pct_under18, population share by borough within quintile:
Borough  Bronx  Brooklyn  Manhattan  Queens  Staten Island
q                                                         
1        0.065     0.219      0.482   0.214          0.020
2        0.109     0.299      0.232   0.317          0.043
3        0.139     0.336      0.140   0.317          0.068
4        0.201     0.326      0.089   0.316          0.068
5        0.299     0.376      0.081   0.194          0.049

med_income, population share by borough within quintile:
Borough  Bronx  Brooklyn  Manhattan  Queens  Staten Island
q                                                         
1        0.374     0.321      0.141   0.145          0.019
2        0.187     0.372      0.119   0.307          0.016
3        0.083     0.337      0.105   0.423          0.052
4        0.066     0.295      0.152   0.390          0.096
5        0.037     0.249      0.463   0.173          0.079


---
## 5. Regressions

The lowest child-share quintile is 48% Manhattan and the highest income quintile is 46% Manhattan, so we add borough fixed effects to separate composition from geography. We model offset level with loss as a covariate, since the offset rate is unstable where loss is near zero, standardize every predictor, and weight by block group population. We report standard errors clustered by tract, since neighboring block groups share Open Streets, alongside HC1 errors for comparison. We enter renter share as a control rather than a variable of interest, since its gradient is not monotonic and appears to proxy density. We refit the reported model on block groups with an income coefficient of variation at or below the 90th percentile, and again without income, to check how much of the racial gradient income absorbs.

In [20]:
d = bg[bg["med_income"].notna() & bg["pct_under18"].notna()].copy()
d["log_income"] = np.log(d["med_income"])
d["offset"] = d[f"offset_{WKEY[-1]}"]
d["tract_geoid"] = d["bg_geoid"].str[:11]

Z = ["pct_nonwhite", "pct_under18", "log_income", "pct_renter", "loss"]
for c in Z:
    d[f"z_{c}"] = (d[c] - d[c].mean()) / d[c].std()

rhs = " + ".join(f"z_{c}" for c in Z)
rhs_no_income = " + ".join(f"z_{c}" for c in Z if c != "log_income")
d2 = d[d["income_cv"] <= d["income_cv"].quantile(0.9)]

s1 = smf.wls(f"offset ~ {rhs}", data=d, weights=d["pop"]).fit(cov_type="HC1")
s2 = smf.wls(f"offset ~ {rhs} + C(Borough)", data=d, weights=d["pop"]).fit(cov_type="HC1")
s3 = smf.wls(f"offset ~ {rhs} + C(Borough)", data=d, weights=d["pop"]).fit(
    cov_type="cluster", cov_kwds={"groups": d["tract_geoid"]})
s4 = smf.wls(f"offset ~ {rhs} + C(Borough)", data=d2, weights=d2["pop"]).fit(
    cov_type="cluster", cov_kwds={"groups": d2["tract_geoid"]})
s5 = smf.wls(f"offset ~ {rhs_no_income} + C(Borough)", data=d, weights=d["pop"]).fit(
    cov_type="cluster", cov_kwds={"groups": d["tract_geoid"]})

MODELS = {"no fixed effects": s1, "borough FE": s2, "borough FE, clustered": s3,
          "borough FE, clustered, reliable income": s4}

coefs = pd.DataFrame({lab: m.params[[f"z_{c}" for c in Z]] for lab, m in MODELS.items()})
print(coefs.round(3).to_string())
print(f"\nR2 {s1.rsquared:.3f} -> {s2.rsquared:.3f}  |  n {int(s1.nobs):,}  |  reliable income n {int(s4.nobs):,}")

print("\nnonwhite share, clustered:")
for lab, m in [("with income", s3), ("without income", s5), ("reliable income", s4)]:
    print(f"  {lab:16} coef {m.params['z_pct_nonwhite']:+.3f}  se {m.bse['z_pct_nonwhite']:.3f}  "
          f"p {m.pvalues['z_pct_nonwhite']:.3f}  95% CI [{m.conf_int().loc['z_pct_nonwhite', 0]:+.3f}, "
          f"{m.conf_int().loc['z_pct_nonwhite', 1]:+.3f}]")

                no fixed effects  borough FE  borough FE, clustered  borough FE, clustered, reliable income
z_pct_nonwhite            -0.086      -0.016                 -0.016                                  -0.019
z_pct_under18             -0.151      -0.130                 -0.130                                  -0.137
z_log_income               0.368       0.327                  0.327                                   0.360
z_pct_renter               0.485       0.388                  0.388                                   0.398
z_loss                     0.552       0.523                  0.523                                   0.504

R2 0.149 -> 0.162  |  n 5,632  |  reliable income n 5,068

nonwhite share, clustered:
  with income      coef -0.016  se 0.041  p 0.689  95% CI [-0.096, +0.064]
  without income   coef -0.136  se 0.052  p 0.008  95% CI [-0.238, -0.035]
  reliable income  coef -0.019  se 0.042  p 0.651  95% CI [-0.101, +0.063]


---
## 6. Export

We write the block group table, the four gradient tables, and the standardized regression coefficients, so `04_figure_formation` builds figures without refitting.

In [22]:
bg.to_csv(PROCESSED / "equity_blockgroup.csv", index=False)

grads = {v: quintile_rates(bg, v) for v in ["pct_nonwhite", "pct_under18", "med_income", "pct_renter"]}
pd.concat(grads, names=["variable"]).to_csv(PROCESSED / "equity_gradients.csv")

out = []
for lab, m in MODELS.items():
    ci = m.conf_int()
    for t in [c for c in m.params.index if c.startswith("z_")]:
        out.append({"model": lab, "term": t, "coef": m.params[t], "se": m.bse[t], "p": m.pvalues[t],
                    "ci_lo": ci.loc[t, 0], "ci_hi": ci.loc[t, 1], "r2": m.rsquared, "n": int(m.nobs)})
pd.DataFrame(out).to_csv(PROCESSED / "equity_regressions_std.csv", index=False)

with open(PROCESSED / "equity_meta.json", "w") as f:
    json.dump({"no_playground_bg": NO_PLAYGROUND_BG,
               "no_playground_pop": NO_PLAYGROUND_POP,
               "bg_analyzed": int(len(bg)),
               "pop_analyzed": float(bg["pop"].sum())}, f, indent=2)

print(f"{len(bg):,} block groups, {len(grads)} gradients, {len(out)} coefficients written")

6,380 block groups, 4 gradients, 20 coefficients written
